# 🎓 AI-Powered Student Performance Analytics Dashboard

## 📌 Project Overview

This project is an end-to-end data science and machine learning application designed to predict and visualize student academic performance. Built using **Google Colab**, **Streamlit**, and **XGBoost**, this dashboard transforms raw student metrics into actionable educational insights, providing a powerful tool for teachers and administrators to identify at-risk students and understand behavioral trends.

Because Google Colab runs on isolated cloud servers, this project utilizes a secure **ngrok tunnel** to safely expose the internal network port, allowing the user to interact with a live, responsive web application directly from their browser.

## 🚀 Key Features

* **🔮 Dynamic GPA Prediction:** A sidebar utility powered by an **XGBoost Regressor** model. Users can manipulate real-time sliders for *Attendance Rate*, *Weekly Study Hours*, and *Past Class Failures* to instantly generate an AI-predicted GPA output.
* **📊 Interactive Analytics & Trends:** Leverages **Plotly Express** to render interactive scatter plots with Ordinary Least Squares (OLS) trendlines, illustrating the mathematical relationship between classroom attendance and final GPAs.
* **🔥 Performance Heatmaps:** Features an auto-calculating correlation matrix heatmap that instantly visually maps out how heavily subject scores (Math, Science, and English) relate to a student's final GPA standing.
* **📈 Distribution Visualizations:** Implements subject-wise box plots to map out score variance, helping educators see where the student cohort is overall succeeding or struggling.
* **🤖 Model Accuracy Tracking:** Includes an evaluation plot graphing *Actual vs. Predicted* metrics to visually prove the underlying machine learning model’s reliability and transparency.


## 🛠️ System Architecture & Workflow

1. **Environment Configuration:** Dependencies (`streamlit`, `pyngrok`, `xgboost`, `scikit-learn`) are cleanly provisioned into the virtual Linux environment.
2. **Synthetic Data Generation:** A balanced dataset of 200 students is synthetically engineered using NumPy random distributions to safely model real-world school metrics without violating data privacy.
3. **Model Training:** The data is split into an 80/20 train/test environment. An Advanced Gradient Boosting (XGBoost) model is fitted against behavioral features to learn predictive weighting.
4. **UI Compiling & Local Deployment:** The Streamlit interface script (`app.py`) is written to disk and initiated as a background process on host port `8501`.
5. **Reverse Proxy Tunneling:** The ngrok client intercepts the local host port and provisions a public, SSL-secured `.ngrok-free.dev` domain link to view the working dashboard UI.


## 📦 Python Libraries Used

* **Core Data Engine:** `pandas`, `numpy`
* **Machine Learning Architecture:** `scikit-learn`, `xgboost`
* **Data Visualization UI:** `plotly`, `streamlit`
* **Cloud Tunneling Proxy:** `pyngrok`

In [1]:
!pip install streamlit plotly xgboost scikit-learn pandas numpy pyngrok --quiet
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


In [2]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)
num_students = 200

# Generate realistic data
attendance = np.random.uniform(60, 100, num_students)
study_hours = np.random.uniform(2, 12, num_students)
failures = np.random.choice([0, 1, 2, 3], num_students, p=[0.8, 0.12, 0.06, 0.02])

# GPA formula with noise
gpa = 1.5 + (attendance * 0.015) + (study_hours * 0.12) - (failures * 0.4) + np.random.normal(0, 0.2, num_students)
gpa = np.clip(gpa, 0.0, 4.0)

# Subject grades (0-100)
math = np.clip(gpa * 22 + np.random.normal(0, 5, num_students), 50, 100)
science = np.clip(gpa * 23 + np.random.normal(0, 4, num_students), 50, 100)
english = np.clip(gpa * 20 + np.random.normal(0, 6, num_students), 50, 100)

df = pd.DataFrame({
    'Student_ID': range(101, 101 + num_students),
    'Attendance_Rate': np.round(attendance, 1),
    'Study_Hours_Weekly': np.round(study_hours, 1),
    'Past_Failures': failures,
    'Math_Score': np.round(math, 1),
    'Science_Score': np.round(science, 1),
    'English_Score': np.round(english, 1),
    'Current_GPA': np.round(gpa, 2)
})

# Save to CSV
df.to_csv('student_data.csv', index=False)
print("✅ 'student_data.csv' created successfully!")

✅ 'student_data.csv' created successfully!


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# Page configuration
st.set_page_config(page_title="Student Analytics Dashboard", layout="wide")
st.title("🎓 AI-Powered Student Performance Analytics")

# Load Dataset
@st.cache_data
def load_data():
    return pd.read_csv('student_data.csv')

df = load_data()

# --- SIDEBAR: ML GPA PREDICTOR ---
st.sidebar.header("🔮 Predict Student GPA")
input_attendance = st.sidebar.slider("Attendance Rate (%)", 60.0, 100.0, 85.0)
input_hours = st.sidebar.slider("Weekly Study Hours", 1.0, 15.0, 7.0)
input_failures = st.sidebar.selectbox("Past Class Failures", [0, 1, 2, 3])

# Train XGBoost Model
X = df[['Attendance_Rate', 'Study_Hours_Weekly', 'Past_Failures']]
y = df['Current_GPA']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3)
model.fit(X_train, y_train)

# Predict based on sidebar inputs
features = np.array([[input_attendance, input_hours, input_failures]])
predicted_gpa = model.predict(features)[0]
st.sidebar.metric(label="Predicted GPA", value=f"{predicted_gpa:.2f} / 4.0")

# --- MAIN DASHBOARD VISUALS ---
col1, col2, col3 = st.columns(3)
col1.metric("Total Students Tracked", len(df))
col2.metric("Average Attendance", f"{df['Attendance_Rate'].mean():.1f}%")
col3.metric("Average GPA", f"{df['Current_GPA'].mean():.2f}")

st.markdown("---")

row1_col1, row1_col2 = st.columns(2)

with row1_col1:
    st.subheader("📊 Attendance vs. GPA Trends")
    fig1 = px.scatter(df, x="Attendance_Rate", y="Current_GPA",
                     trendline="ols", color="Study_Hours_Weekly",
                     labels={"Attendance_Rate": "Attendance (%)", "Current_GPA": "GPA"})
    st.plotly_chart(fig1, use_container_width=True)

with row1_col2:
    st.subheader("🔥 Performance Correlation Heatmap")
    subjects = ['Math_Score', 'Science_Score', 'English_Score', 'Current_GPA']
    corr_matrix = df[subjects].corr()
    fig2 = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdBu_r')
    st.plotly_chart(fig2, use_container_width=True)

row2_col1, row2_col2 = st.columns(2)

with row2_col1:
    st.subheader("📈 Subject Score Distributions")
    df_melted = df.melt(id_vars=['Student_ID'], value_vars=['Math_Score', 'Science_Score', 'English_Score'],
                        var_name='Subject', value_name='Score')
    fig3 = px.box(df_melted, x='Subject', y='Score', color='Subject')
    st.plotly_chart(fig3, use_container_width=True)

with row2_col2:
    st.subheader("🤖 Model Accuracy (Actual vs. Predicted)")
    y_pred = model.predict(X_test)
    eval_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
    fig4 = px.scatter(eval_df, x='Actual', y='Predicted', trendline="ols")
    st.plotly_chart(fig4, use_container_width=True)

if st.checkbox("Show Data Log"):
    st.dataframe(df)

Overwriting app.py


In [9]:
import os
from pyngrok import ngrok

# 1. Terminate any residual background processes
ngrok.kill()

# 2. REPLACE THE XXXXXX BELOW WITH YOUR ACTUAL NGROK TOKEN.
# Keep the single quotes around it!
REAL_TOKEN = '3EMUT2tRQi9STwsmgFQGKMw3fK1_4yKb6mGiaepAQUbhMX64B'

# 3. Force-write the token directly into the system's terminal configuration
!ngrok config add-authtoken {REAL_TOKEN}

# 4. Spin up your Streamlit dashboard in the background
os.system("streamlit run app.py &")

# 5. Open the tunnel connection
try:
    public_url = ngrok.connect(8501)
    print("\n🚀 SUCCESS! Your dashboard is now alive.")
    print(f"🔗 Click this link to open it: {public_url.public_url}")
except Exception as e:
    print(f"❌ Connection Error: {e}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

🚀 SUCCESS! Your dashboard is now alive.
🔗 Click this link to open it: https://hermit-badness-kennel.ngrok-free.dev
